# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Use the dataset object to inspect record sets and fields
croissant_json = dataset.metadata.to_json()

# Extract and print available record sets by @id
record_sets = croissant_json.get('recordSet', [])
if not record_sets:
    print("No record sets found in metadata. Attempting inference from data files...")
    # If record sets not specified, try to get from distribution
    distributions = croissant_json.get('distribution', [])
    print("Distributions in dataset:")
    for dist in distributions:
        print("  @id:", dist.get('@id'))
else:
    print("Record Sets in dataset:")
    for rs in record_sets:
        if isinstance(rs, dict) and '@id' in rs:
            print("  RecordSet @id:", rs['@id'])
        elif isinstance(rs, str):
            print("  RecordSet @id:", rs)

# If distributions contain CSV data, let's explore fields
# Let's look at the first distribution's contentUrl (if available)
fields = croissant_json.get('field', [])
if fields:
    print("Fields in dataset:")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"  Field @id: {field['@id']} | name: {field.get('name')} | dataType: {field.get('dataType')}")
        elif isinstance(field, str):
            print(f"  Field @id: {field}")
else:
    print("No fields found in metadata. Fields can be inferred after loading records.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, recordSets are not explicitly listed in metadata, so
# We'll load data from the main dataset record set (the top-level dataset @id)
# Alternatively, if available, use distribution @ids as record sets

# Use the dataset @id as 'record_set'
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/e1f3c048-91dc-444a-b31e-e649e1eb302f'

# Or, get available distribution @ids to try loading records
distribution_ids = [dist.get('@id') for dist in metadata.get('distribution', []) if '@id' in dist]
record_sets = [main_record_set_id] + distribution_ids

# Store DataFrames by record set @id
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded records for record set @id: {record_set}")
            print(f"Columns: {dataframes[record_set].columns.tolist()}")
        else:
            print(f"No records returned for record set @id: {record_set}")
    except Exception as e:
        print(f"Failed to load records for record set @id: {record_set} | Error: {e}")

# Print columns and show first few rows for primary record set
primary_rs = main_record_set_id if main_record_set_id in dataframes else (distribution_ids[0] if distribution_ids and distribution_ids[0] in dataframes else None)
if primary_rs:
    print("Columns in primary record set:", dataframes[primary_rs].columns.tolist())
    display(dataframes[primary_rs].head())
else:
    print("No DataFrame available for primary record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select record set and fields for EDA
record_set_id = primary_rs
df = dataframes.get(record_set_id)

# Try to identify numeric fields for analysis from available columns
if df is not None:
    numeric_field_candidates = [col for col in df.columns if ('score' in col.lower() or 'age' in col.lower() or df[col].dtype in [float, int])]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field for filtering: '{numeric_field}'")

        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by categorical field, e.g. 'gender' or 'level_of_education'
        cat_candidates = [col for col in df.columns if col.lower() in ['gender', 'level_of_education', 'village', 'marital_status']]
        group_field = cat_candidates[0] if cat_candidates else df.columns[0]
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print(f"No DataFrame available for EDA for record set @id: {record_set_id}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization: Histogram and Boxplot for numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Box plot grouped by categorical field
    if group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data or numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We obtained metadata, inspected the structure of the dataset, used `@id` references for record sets and fields, and performed basic exploratory data analysis—including filtering, normalization, grouping, and visualization. The dataset contains rich information about demographics and mental health indicators. Further analyses can focus on modeling risk factors, tracing cohort subgroups, and comparing assessment scores across key variables.